# 02 · Self-contained HTML — globes & 2D maps

`da.tp.to_html(path, ...)` writes a single **self-contained** `.html` file: the terraplot JS bundle,
your data (gzip-compressed binary by default), a colorbar, and a label, all inlined. No server, no build step —
open it in any modern browser (or embed it in a notebook via `IFrame`).

- `projection=None` (default) → **3D globe** (`GeoSphere`)
- a projection name → **2D flat map** (`GeoMap`) with hover tooltips

We render results inline with `IFrame`. The HTML needs the terraplot bundle (auto-detected sibling repo
or `TERRAPLOT_BUNDLE`).

In [ ]:
import numpy as np
import xarray as xr
import pyterraplot  # registers the .tp accessor on DataArray and Dataset

def make_field(nlat=73, nlon=144, phase=0.0, name="t2m",
               long_name="2m temperature anomaly", units="K", holes=True):
    """A smooth, globe-shaped synthetic field on a regular lat/lon grid."""
    lats = np.linspace(90, -90, nlat)
    lons = np.linspace(-180, 180, nlon)
    LON, LAT = np.meshgrid(lons, lats)
    data = (
        8 * np.cos(np.radians(LAT)) * np.sin(np.radians(2 * LON) + phase)
        + 5 * np.sin(np.radians(3 * LON)) * np.cos(np.radians(2 * LAT))
        + 3 * np.cos(np.radians(5 * LON)) * np.sin(np.radians(LAT))
    ).astype(np.float32)
    if holes:
        rng = np.random.default_rng(0)
        data[rng.random((nlat, nlon)) < 0.02] = np.nan  # NaN "missing" cells
    return xr.DataArray(
        data, dims=["lat", "lon"], coords={"lat": lats, "lon": lons},
        name=name, attrs={"units": units, "long_name": long_name},
    )

da = make_field()
da

In [ ]:
from IPython.display import IFrame
from pathlib import Path

def show(path, h=420):
    kb = Path(path).stat().st_size / 1024
    print(f"{path}  ({kb:.0f} kB)")
    return IFrame(path, width="100%", height=h)

## 3D globe (default)

In [ ]:
da.tp.to_html("globe.html", title="3D globe", cmap="RdYlBu_r", alpha=0.85)
show("globe.html")

## `contourf` vs `pcolormesh`

`kind='pcolormesh'` (default) is smooth; `kind='contourf'` draws `levels` discrete bands. `vmin`/`vmax` pin the color range (auto from data if omitted).

In [ ]:
da.tp.to_html("contourf.html", kind="contourf", levels=14,
              cmap="viridis", vmin=-12, vmax=12, title="contourf 14 levels")
show("contourf.html")

## 2D projection map

Pass a `projection`. Supported: `equirectangular` (aka `PlateCarree`), `mercator`, `orthographic`, `naturalEarth`, `stereographic`, `azimuthalEqualArea`, `albers`, `lambertConformal`, `gnomonic`. 2D maps get interactive hover tooltips (lat/lon/value) and optional `coastlines`.

In [ ]:
da.tp.to_html("natural_earth.html", projection="naturalEarth",
              cmap="RdYlBu_r", coastlines=True, title="Natural Earth")
show("natural_earth.html")

## Regional zoom with `extent` + `center`

`extent=(lon0, lon1, lat0, lat1)` works like cartopy's `set_extent`; `center=(lon, lat)` recenters the projection. Here: the North Atlantic on a Lambert map.

In [ ]:
da.tp.to_html("region.html", projection="lambertConformal",
              center=(-40, 50), extent=(-80, 0, 20, 70),
              cmap="RdBu_r", title="North Atlantic")
show("region.html")

## JSON vs binary payload

`binary=True` (default) inlines gzip-compressed float32 (needs `DecompressionStream` — Chrome 80+/FF 113+/Safari 16.4+). `binary=False` inlines plain JSON: larger, but works everywhere and is human-readable. Compare the file sizes:

In [ ]:
bpath = da.tp.to_html("payload_binary.html", binary=True)
jpath = da.tp.to_html("payload_json.html",   binary=False)
kb = lambda p: Path(p).stat().st_size / 1024
print(f"binary : {kb(bpath):6.0f} kB")
print(f"json   : {kb(jpath):6.0f} kB")
print(f"ratio  : {kb(jpath)/kb(bpath):.1f}× larger as JSON")